In [138]:
import os
from dotenv import load_dotenv, find_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [139]:
print(os.getcwd())
print(load_dotenv(find_dotenv(usecwd=True), override=True))


/Users/zhaojiayi/Desktop/rag_studymemo/01_rag_basic
True


In [140]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")


In [141]:
DATA_FILE_PATH= os.path.join("data","hr_policy.txt")

In [142]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probatio

In [143]:
print(f"Total characters in document:{len(documents[0].page_content)}")

Total characters in document:2598


In [144]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size = 500,
  chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)
len(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

9

In [145]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v5-omni-nano")
print("EMB model ready the name is",embeddings_model.model_name)

EMB model ready the name is jina-embeddings-v5-omni-nano


In [146]:
vector_store =FAISS.from_documents(chunks,embeddings_model)
print("Chunks are stored",vector_store.index.ntotal)

Chunks are stored 9


In [147]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query,k=2)
print(f"Query:{test_query}")
for i,match in enumerate(top_matches,start = 1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query:How many sick leaves employees get
--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [148]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [149]:
llm = ChatGroq(
  model="openai/gpt-oss-120b",
  temperature=0
)

llm.model_name

'openai/gpt-oss-120b'

In [150]:
test_response = llm.invoke("hey is learning Rag hard? reply in a joke manner,in one line")

test_response.content

'Learning RAG is like trying to teach a cat to fetch—possible, but you’ll spend most of the time chasing after the “retrieval” part!'

In [151]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """You are a friendly HR assistant working for Acme Corp.
Always use the search_hr_policy tool to look up facts before answering.
If the answer isn't in the search results, say you don't know instead of guessing."""
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [152]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [153]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [154]:
ask_hr_assistant("How many sick leaves do employees get?")
ask_hr_assistant("What is the notice period during probation?")
ask_hr_assistant("What is the CEO's salary?")  # 文档里没有，应回答不知道

QUESTION: How many sick leaves do employees get?
------------------------------------------------------------
ANSWER: Employees are entitled to **10 paid sick days per year**.

QUESTION: What is the notice period during probation?
------------------------------------------------------------
ANSWER: During the probation period, the required notice period is **15 days**.

QUESTION: What is the CEO's salary?
------------------------------------------------------------
ANSWER: I’m sorry, but I don’t have that information.



'I’m sorry, but I don’t have that information.'